# GraphSAE Minimal Reproduction on Cora

This notebook executes the submission-oriented Cora pipeline: backbone training, last-layer activation extraction, SAE training, feature alignment, SAE-GNNExplainer, and original-graph masking. Run it from the `paper_reproduction` directory.

In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd()
assert (ROOT / 'scripts').exists(), 'Open this notebook with paper_reproduction as the working directory.'
MODEL = 'GCN'  # Change to GAT, GIN, or GraphSAGE for the other backbones.
DEVICE = 'auto'


## 1. Train a capture-compatible backbone

The final message-passing representation is captured after ReLU for GCN/GIN/GraphSAGE and after ELU for GAT.

In [ ]:
!python scripts/01_train_cora_backbone.py --model {MODEL} --epochs 300 --device {DEVICE}

## 2. Extract final-layer activations and train the SAE

In [ ]:
!python scripts/02_extract_activations.py --model {MODEL} --split train --device {DEVICE}
!python scripts/03_train_sae.py --model {MODEL} --activation-split train --epochs 200 --device {DEVICE}

## 3. Semantic and causal feature alignment

This writes a purity/activity table and ablates the selected target node's active SAE features through a last-layer patch.

In [ ]:
!python scripts/04_feature_alignment.py --model {MODEL} --device {DEVICE}
alignment = pd.read_csv(ROOT / 'artifacts' / 'cora' / f'feature_alignment_{MODEL.lower()}.csv')
alignment.head(10)

## 4. SAE-GNNExplainer and mask verification

The explainer writes SAE+, SAE-, and SAE+/- node groups. The final command evaluates feature-and-edge masking for each group.

In [ ]:
!python scripts/05_sae_gnnexplainer.py --model {MODEL} --top-k 8 --device {DEVICE}
!python scripts/06_mask_experiment.py --model {MODEL} --top-k 8 --mask-mode both --device {DEVICE}

The result JSON files are in `artifacts/cora`. To run the graph-classification path, execute `python scripts/07_graph_classification_demo.py --dataset MUTAG --model GIN`.